<a href="https://colab.research.google.com/github/manikanta-eng/HPC/blob/main/hpc_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import multiprocessing as mp
import time
import random

class Patient:
    def __init__(self, pid):
        self.id = pid
        self.heart_rate = random.randint(50, 130)
        self.temperature = round(random.uniform(35, 40), 1)
        self.oxygen = random.randint(80, 100)

def evaluate_patient(patient):
    alerts = []
    score = 0

    if patient.heart_rate < 60 or patient.heart_rate > 100:
        alerts.append("Abnormal Heart Rate")
        score += 2

    if patient.temperature > 37.5:
        alerts.append("High Temperature")
        score += 2

    if patient.oxygen < 95:
        alerts.append("Low Oxygen Level")
        score += 3

    if score >= 5:
        status = "CRITICAL"
    elif score >= 2:
        status = "WARNING"
    else:
        status = "NORMAL"

    return {
        "id": patient.id,
        "heart_rate": patient.heart_rate,
        "temperature": patient.temperature,
        "oxygen": patient.oxygen,
        "status": status,
        "alerts": alerts,
        "score": score
    }

def sequential_processing(patients):
    results = []
    for p in patients:
        results.append(evaluate_patient(p))
    return results

def parallel_processing(patients):
    pool = mp.Pool(mp.cpu_count())
    results = pool.map(evaluate_patient, patients)
    pool.close()
    pool.join()
    return results

def generate_statistics(results):
    total = len(results)
    normal = sum(1 for r in results if r["status"] == "NORMAL")
    warning = sum(1 for r in results if r["status"] == "WARNING")
    critical = sum(1 for r in results if r["status"] == "CRITICAL")

    avg_hr = sum(r["heart_rate"] for r in results) / total
    avg_temp = sum(r["temperature"] for r in results) / total
    avg_oxygen = sum(r["oxygen"] for r in results) / total

    return {
        "total": total,
        "normal": normal,
        "warning": warning,
        "critical": critical,
        "avg_hr": round(avg_hr, 2),
        "avg_temp": round(avg_temp, 2),
        "avg_oxygen": round(avg_oxygen, 2)
    }

if __name__ == "__main__":
    n = int(input("Enter number of patients: "))

    patients = [Patient(i) for i in range(n)]

    start = time.time()
    seq_results = sequential_processing(patients)
    seq_time = time.time() - start

    start = time.time()
    par_results = parallel_processing(patients)
    par_time = time.time() - start

    stats = generate_statistics(par_results)

    print("\n--- PATIENT REPORT ---")
    for r in par_results:
        print(f"Patient {r['id']} | HR:{r['heart_rate']} | Temp:{r['temperature']} | O2:{r['oxygen']}")
        print(f"Status: {r['status']} | Alerts: {r['alerts']} | Score: {r['score']}")
        print("-"*50)

    print("\n--- SYSTEM STATISTICS ---")
    print(f"Total Patients: {stats['total']}")
    print(f"Normal: {stats['normal']}")
    print(f"Warning: {stats['warning']}")
    print(f"Critical: {stats['critical']}")
    print(f"Avg Heart Rate: {stats['avg_hr']}")
    print(f"Avg Temperature: {stats['avg_temp']}")
    print(f"Avg Oxygen Level: {stats['avg_oxygen']}")

    print("\n--- PERFORMANCE ---")
    print(f"Sequential Time: {round(seq_time,4)} sec")
    print(f"Parallel Time: {round(par_time,4)} sec")

Enter number of patients: 5

--- PATIENT REPORT ---
Patient 0 | HR:81 | Temp:38.1 | O2:92
Status: CRITICAL | Alerts: ['High Temperature', 'Low Oxygen Level'] | Score: 5
--------------------------------------------------
Patient 1 | HR:130 | Temp:38.4 | O2:89
Status: CRITICAL | Alerts: ['Abnormal Heart Rate', 'High Temperature', 'Low Oxygen Level'] | Score: 7
--------------------------------------------------
Patient 2 | HR:103 | Temp:37.2 | O2:91
Status: CRITICAL | Alerts: ['Abnormal Heart Rate', 'Low Oxygen Level'] | Score: 5
--------------------------------------------------
Patient 3 | HR:63 | Temp:39.6 | O2:93
Status: CRITICAL | Alerts: ['High Temperature', 'Low Oxygen Level'] | Score: 5
--------------------------------------------------
Patient 4 | HR:53 | Temp:36.7 | O2:100
Status: WARNING | Alerts: ['Abnormal Heart Rate'] | Score: 2
--------------------------------------------------

--- SYSTEM STATISTICS ---
Total Patients: 5
Normal: 0
Critical: 4
Avg Heart Rate: 86.0
Avg Tempe